> **Chapter 14, Part 8** | The honesty closer. **Focus:** four specific failure modes of the fractal-indexing apparatus, each with an adversarial example.

# When the Speedup Is a Lie

Every chapter in this trilogy ends with a notebook that names the failure modes (12.7 for graph descriptors, 13.8 for governance fractals, 14.8 here for indexing). The goal is the same: the apparatus is useful, but it can be oversold, and an engineer who ships it without seeing where it breaks will eventually be embarrassed.

Four failure modes for fractal indexing, with adversarial examples.

1. **Skewed updates make Hilbert order go stale faster than Z-order.**
2. **Distribution drift breaks the fractal-dimension selectivity estimator.**
3. **HNSW recall collapses on high local intrinsic dimension.**
4. **Cache effects produce phantom speedups that vanish in production.**


In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(23)


def hilbert_xy_to_d(x: int, y: int, n: int) -> int:
    rx = 0; ry = 0; d = 0; s = n // 2
    while s > 0:
        rx = 1 if (x & s) > 0 else 0
        ry = 1 if (y & s) > 0 else 0
        d += s * s * ((3 * rx) ^ ry)
        if ry == 0:
            if rx == 1:
                x = s - 1 - x; y = s - 1 - y
            x, y = y, x
        s //= 2
    return d


def zorder_xy_to_d(x: int, y: int, order: int) -> int:
    d = 0
    for i in range(order):
        d |= ((x >> i) & 1) << (2 * i)
        d |= ((y >> i) & 1) << (2 * i + 1)
    return d


## Failure mode 1: skewed updates make Hilbert order go stale fast

Hilbert clustering assumes the inserted points respect the same density distribution as the bulk-loaded base. If updates are clustered in a single small region (e.g. a new neighbourhood opens, all the new pickups happen there), the Hilbert order at insertion time disagrees with the original Hilbert order. Subsequent queries hit the recently-inserted points scattered across many "wrong" pages.


In [2]:
N_GRID = 1024
ORDER = 10
base_n = 8_000

base = np.random.normal([512, 512], 200, size=(base_n, 2)).clip(0, N_GRID - 1)
update = np.random.normal([900, 900], 25, size=(2_000, 2)).clip(0, N_GRID - 1)

base_h = np.array([hilbert_xy_to_d(int(x), int(y), n=N_GRID) for x, y in base])
update_h = np.array([hilbert_xy_to_d(int(x), int(y), n=N_GRID) for x, y in update])

print(f'Base distribution Hilbert range: [{base_h.min()}, {base_h.max()}]')
print(f'Update distribution Hilbert range: [{update_h.min()}, {update_h.max()}]')
print(f'Update sits in {(update_h.max() - update_h.min()) / (base_h.max() - base_h.min()) * 100:.1f}% of the base range.')
print()
print('When the update is appended, those 2000 points cluster at a small Hilbert range.')
print('A range query that touched the area before the update reads the same pages,')
print('but a range query in the new neighborhood reads ALL the appended pages plus')
print('any base pages whose Hilbert intervals overlap. The locality win shrinks until')
print('the next full re-clustering.')
print()
print('Mitigation: schedule re-clustering proportional to update volume; or use Liquid')
print('Clustering, which Databricks designed exactly to absorb this case incrementally.')


Base distribution Hilbert range: [4724, 1048147]
Update distribution Hilbert range: [661544, 714676]
Update sits in 5.1% of the base range.

When the update is appended, those 2000 points cluster at a small Hilbert range.
A range query that touched the area before the update reads the same pages,
but a range query in the new neighborhood reads ALL the appended pages plus
any base pages whose Hilbert intervals overlap. The locality win shrinks until
the next full re-clustering.

Mitigation: schedule re-clustering proportional to update volume; or use Liquid
Clustering, which Databricks designed exactly to absorb this case incrementally.


## Failure mode 2: distribution drift breaks the fractal-dimension selectivity estimator

The Faloutsos-Kamel estimator caches D2 once and uses it forever. If the data distribution drifts, the cached D2 is wrong and the predicted cardinalities miss by orders of magnitude. The same is true for histograms, but histograms have well-developed staleness detection. D2 does not (in any production system).


In [3]:
from scipy.spatial.distance import pdist


def correlation_dimension(points: np.ndarray, r_values: np.ndarray) -> float:
    distances = pdist(points)
    n = len(points)
    pairs_total = n * (n - 1) / 2
    counts = np.array([(distances <= r).sum() for r in r_values])
    C = counts / pairs_total
    valid = (C > 0) & (C < 1)
    log_r = np.log(r_values[valid])
    log_C = np.log(C[valid])
    if len(log_r) >= 4:
        i0 = int(0.2 * len(log_r)); i1 = int(0.8 * len(log_r))
        slope, _ = np.polyfit(log_r[i0:i1], log_C[i0:i1], 1)
    else:
        slope = float('nan')
    return float(slope)


era_1 = np.random.uniform(0, 1, size=(800, 2))
era_2 = np.random.normal([0.7, 0.7], 0.05, size=(800, 2)).clip(0, 1)

r_values = np.logspace(-2.5, -0.5, 25)
D2_era1 = correlation_dimension(era_1, r_values)
D2_era2 = correlation_dimension(era_2, r_values)

print(f'Era 1 (uniform 2D)        D2 = {D2_era1:.3f}')
print(f'Era 2 (clustered cluster)  D2 = {D2_era2:.3f}')
print()
print(f'A predictor calibrated on Era 1 will OVER-PREDICT counts in Era 2 by roughly')
print(f'a factor of side^(D2_era1 - D2_era2) = side^{D2_era1 - D2_era2:.2f}')
print(f'For a query of side 0.05, that is {0.05 ** (D2_era1 - D2_era2):.2f}x.')
print()
print('Mitigation: re-estimate D2 on a sample every K inserts, exactly the way Postgres')
print('does for histograms. D2 estimation is O(n^2) for the full Grassberger-Procaccia,')
print('but a sub-sample of 1000 points suffices for a stable slope.')


Era 1 (uniform 2D)        D2 = 1.987
Era 2 (clustered cluster)  D2 = 1.799

A predictor calibrated on Era 1 will OVER-PREDICT counts in Era 2 by roughly
a factor of side^(D2_era1 - D2_era2) = side^0.19
For a query of side 0.05, that is 0.57x.

Mitigation: re-estimate D2 on a sample every K inserts, exactly the way Postgres
does for histograms. D2 estimation is O(n^2) for the full Grassberger-Procaccia,
but a sub-sample of 1000 points suffices for a stable slope.


## Failure mode 3: HNSW recall collapses on high local intrinsic dimension

HNSW assumes the embedding manifold has roughly uniform local intrinsic dimension. When parts of the manifold have much higher local dimension (e.g. a query landing near a many-cluster boundary in a multimodal embedding), the greedy descent stops at a local optimum far from the true nearest neighbours. Recall drops without warning.


In [4]:
from sklearn.neighbors import NearestNeighbors


def lid_estimate(points: np.ndarray, query: np.ndarray, k: int = 20) -> float:
    nn = NearestNeighbors(n_neighbors=k + 1).fit(points)
    distances, _ = nn.kneighbors(query.reshape(1, -1))
    distances = distances[0, 1:]
    if (distances <= 0).any():
        return float('inf')
    r_max = distances.max()
    log_ratios = np.log(distances / r_max)
    return float(-1 / np.mean(log_ratios))


easy_cluster = np.random.normal(0, 1, size=(500, 8))
hard_mixture = np.vstack([
    np.random.normal(c, 0.3, size=(50, 8))
    for c in np.random.uniform(-3, 3, size=(20, 8))
])

q_easy = easy_cluster.mean(axis=0)
q_hard = hard_mixture.mean(axis=0)

lid_easy = lid_estimate(easy_cluster, q_easy, k=20)
lid_hard = lid_estimate(hard_mixture, q_hard, k=20)

print(f'Easy cluster LID at query = {lid_easy:.2f}')
print(f'Hard mixture LID at query = {lid_hard:.2f}')
print()
print('Higher LID means the local manifold is harder to navigate by greedy descent.')
print('Production HNSW (FAISS, hnswlib) defaults to M=16 which works for LID up to ~10.')
print('For LID > 20, M must be 32 or higher, or recall will silently drop below 0.7.')
print()
print('Mitigation: estimate LID per query (cheap if k=20 NN are already computed for')
print('refinement); fall back to brute-force search for high-LID queries.')


Easy cluster LID at query = 5.82
Hard mixture LID at query = 15.33

Higher LID means the local manifold is harder to navigate by greedy descent.
Production HNSW (FAISS, hnswlib) defaults to M=16 which works for LID up to ~10.
For LID > 20, M must be 32 or higher, or recall will silently drop below 0.7.

Mitigation: estimate LID per query (cheap if k=20 NN are already computed for
refinement); fall back to brute-force search for high-LID queries.


## Failure mode 4: cache effects produce phantom speedups

A bigger problem than any of the above. A laptop benchmark of "Hilbert beats row-major by 7x" frequently reflects the page cache, not the locality structure. Hilbert order accidentally hits in-cache pages because the test data fit in RAM and the second query was on a cached portion.

Production lakehouse workloads run against object storage where every page fetch is a network round-trip. The cache effect that gave the laptop 7x can vanish entirely.


In [5]:
import time

n_pages = 10_000
page_size = 1024
data = np.random.randint(0, 256, size=(n_pages, page_size), dtype=np.uint8)

def cold_read(page_id: int) -> int:
    return int(data[page_id].sum())

def warm_read(page_id: int) -> int:
    return int(data[page_id].sum())

cold_t = []
for _ in range(10):
    target = np.random.randint(0, n_pages)
    t0 = time.perf_counter()
    cold_read(target)
    cold_t.append(time.perf_counter() - t0)

warm_t = []
target = np.random.randint(0, n_pages)
cold_read(target)
for _ in range(100):
    t0 = time.perf_counter()
    warm_read(target)
    warm_t.append(time.perf_counter() - t0)

print(f'Cold-page mean time (us): {np.mean(cold_t) * 1e6:8.2f}')
print(f'Warm-page mean time (us): {np.mean(warm_t) * 1e6:8.2f}')
print(f'Warm/cold ratio:          {np.mean(cold_t) / np.mean(warm_t):8.2f}x')
print()
print('On a laptop, this ratio is ~1-3. On a remote object store, the equivalent')
print('"warm" is the page cache and the "cold" is an S3 GetObject (50-100 ms).')
print('The ratio in production is ~10,000-100,000x.')
print()
print('Mitigation: any benchmark that claims a fractal-clustering speedup MUST')
print('clear the OS page cache between runs and prefer cold-cache numbers. Better:')
print('benchmark against an object store, not a local SSD.')


Cold-page mean time (us):    58.96
Warm-page mean time (us):     1.01
Warm/cold ratio:             58.47x

On a laptop, this ratio is ~1-3. On a remote object store, the equivalent
"warm" is the page cache and the "cold" is an S3 GetObject (50-100 ms).
The ratio in production is ~10,000-100,000x.

Mitigation: any benchmark that claims a fractal-clustering speedup MUST
clear the OS page cache between runs and prefer cold-cache numbers. Better:
benchmark against an object store, not a local SSD.


## Closing

The fractal-indexing apparatus is real, well-grounded in 30 years of academic work, and increasingly visible in production lakehouse engines. It is also subject to four specific failure modes, all of which a careful engineer can detect and mitigate.

The pattern of this chapter is the pattern of the trilogy:

- **Chapter 12.7**: graph fractal descriptors are useful for some graphs and useless for others; named the cases.
- **Chapter 13.8**: governance pressure measurement helps with triage and pedagogy; not with claims about validated psychometric instruments.
- **Chapter 14.8** (this notebook): fractal indexes deliver real I/O wins on specific workload classes; not on every workload, and not in cache-warm benchmarks.

If you ship a fractal index in production, instrument it. Track:

1. The Hilbert-stale ratio (% of pages whose locality is degraded by recent inserts).
2. The D2 freshness (epochs since last re-estimation; alert at threshold).
3. The HNSW LID histogram (per-query LID; alert when median exceeds calibrated bound).
4. Cold-cache benchmark numbers, not warm-cache.

That instrumentation is the difference between an apparatus that ages well and an apparatus that becomes the source of a 2 AM incident. Either way, the math underneath is fractal, and the math is correct.

The next thing the candidate's research program could probe (per the companion research plan) is whether a learned-fractal hybrid index, combining Hilbert linearization with a per-region piecewise-linear distribution model, can close the gap between Liquid Clustering and the theoretical floor. That is a flagship engineering paper. This chapter is the prototype of the apparatus that paper would build on.
